# CRUD en Python con SQLAlchemy ORM
Este cuaderno muestra cómo realizar operaciones CRUD utilizando el ORM `SQLAlchemy`, que permite trabajar con bases de datos como si fueran objetos de Python.

**<span style="color:red">IMPORTANTE</span>**

Recientemente se ha publicado la versión 2.0 de SQLAlchemy, por lo que los apartados 2-9 están actualizados a la nueva forma de actuar. Se ha mantenido la información relativa a la forma antigua a modo informativo.



## Introducción
- Un **ORM** (Object Relational Mapper) facilita el trabajo con bases de datos.
- SQLAlchemy permite definir **modelos** como clases de Python.
- Vamos a trabajar con una tabla `Usuario` con los campos: `id`, `nombre`, `edad`.

# SQLAlchemy 2.0

## 2. Configuración e instalación
Asegúrate de tener instalado SQLAlchemy:


In [ ]:
pip install sqlalchemy

In [ ]:
from sqlalchemy import create_engine, String, Float, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

# Crear la base de datos
class Base(DeclarativeBase):
    pass

## 3. Definir nuestra primera tabla como clase Python

In [ ]:
class Producto(Base):
    __tablename__ = 'productos'
    
    # Con type annotations (recomendado)
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre: Mapped[str] = mapped_column(String(100), nullable=False)
    precio: Mapped[float] = mapped_column(Float, nullable=False)
    categoria: Mapped[str] = mapped_column(String(50))
    
    def __repr__(self):
        return f"<Producto(nombre='{self.nombre}', precio={self.precio})>"

print("Clase Producto creada")

## 4. Conectar y crear la base de datos

In [ ]:
# Crear motor de base de datos (ACTUALIZADO - sin sessionmaker)
engine = create_engine('sqlite:///tienda.db', echo=True)

# Crear todas las tablas
Base.metadata.create_all(engine)

print("Base de datos creada")

## 5. Insertar datos (como objetos Python)


In [ ]:
# Insertar productos iniciales
with Session(engine) as session:
    productos_iniciales = [
        Producto(nombre="Laptop", precio=899.99, categoria="Tecnología"),
        Producto(nombre="Mouse", precio=25.50, categoria="Tecnología"),
        Producto(nombre="Libro Python", precio=35.00, categoria="Libros"),
        Producto(nombre="Café", precio=5.75, categoria="Alimentos")
    ]

    # Agregar a la sesión y guardar
    session.add_all(productos_iniciales)
    session.commit()

    print("Productos insertados en la base de datos")

## 6. Consultar datos

In [ ]:
with Session(engine) as session:
    # Obtener todos los productos
    print("📦 Todos los productos:")
    stmt = select(Producto)
    productos = session.scalars(stmt).all()
    for producto in productos:
        print(f" - {producto.nombre}: ${producto.precio}")

    # Consultas con filtros
    print("\n💻 Productos de tecnología:")
    stmt_tecno = select(Producto).where(Producto.categoria == "Tecnología")
    tecno = session.scalars(stmt_tecno).all()
    for producto in tecno:
        print(f" - {producto.nombre}")

    # Productos caros (más de $30)
    print("\n💰 Productos caros (> $30):")
    stmt_caros = select(Producto).where(Producto.precio > 30)
    caros = session.scalars(stmt_caros).all()
    for producto in caros:
        print(f" - {producto.nombre}: ${producto.precio}")
        
    #MÁS TIPOS DE CONSULTAS
    # Consulta con múltiples condiciones
    print("\n🖥️ Productos de tecnología caros:")
    stmt_tecno_caros = select(Producto).where(
        (Producto.categoria == "Tecnología") & 
        (Producto.precio > 50)
    )
    tecno_caros = session.scalars(stmt_tecno_caros).all()
    for producto in tecno_caros:
        print(f" - {producto.nombre}: ${producto.precio}")

    # Ordenar resultados
    print("\n📊 Productos ordenados por precio:")
    stmt_ordenados = select(Producto).order_by(Producto.precio.desc())
    ordenados = session.scalars(stmt_ordenados).all()
    for producto in ordenados:
        print(f" - {producto.nombre}: ${producto.precio}")

    # Obtener solo el primer resultado
    print("\n🥇 Producto más barato:")
    stmt_barato = select(Producto).order_by(Producto.precio.asc())
    producto_barato = session.scalar(stmt_barato)
    print(f" - {producto_barato.nombre}: ${producto_barato.precio}")


## 7. Actualizar y eliminar

In [ ]:
with Session(engine) as session:
    # Actualizar un producto
    print("\n🔄 Actualizando producto...")
    stmt_laptop = select(Producto).where(Producto.nombre == "Laptop")
    laptop = session.scalar(stmt_laptop)
    if laptop:
        laptop.precio = 799.99  # ¡Oferta!
        session.commit()
        print(f"✅ Precio de {laptop.nombre} actualizado a ${laptop.precio}")

    # Eliminar un producto
    print("\n🗑️ Eliminando producto...")
    stmt_cafe = select(Producto).where(Producto.nombre == "Café")
    cafe = session.scalar(stmt_cafe)
    if cafe:
        session.delete(cafe)
        session.commit()
        print(f"✅ {cafe.nombre} eliminado")

    # Verificar cambios
    print("\n📋 Productos actualizados:")
    stmt_actualizados = select(Producto)
    productos_actualizados = session.scalars(stmt_actualizados).all()
    for producto in productos_actualizados:
        print(f" - {producto.nombre}: ${producto.precio}")
        
    #MÁS EJEMPLOS
    # Actualizar múltiples productos
    print("\n🎯 Aplicando descuento a productos de tecnología...")
    stmt_tecno = select(Producto).where(Producto.categoria == "Tecnología")
    productos_tecno = session.scalars(stmt_tecno).all()
    for producto in productos_tecno:
        producto.precio = producto.precio * 0.9  # 10% de descuento
        print(f"💰 {producto.nombre} ahora: ${producto.precio:.2f}")
    session.commit()

    # Eliminar con condición directa
    print("\n🧹 Eliminando productos baratos...")
    stmt_baratos = select(Producto).where(Producto.precio < 10)
    productos_baratos = session.scalars(stmt_baratos).all()
    for producto in productos_baratos:
        session.delete(producto)
        print(f"✅ Eliminado: {producto.nombre}")
    session.commit()

    # Contar productos
    from sqlalchemy import func
    print("\n📊 Estadísticas:")
    total_productos = session.scalar(select(func.count(Producto.id)))
    print(f"Total de productos: {total_productos}")

    precio_promedio = session.scalar(select(func.avg(Producto.precio)))
    print(f"Precio promedio: ${precio_promedio:.2f}")
        

## 7.Ejemplo práctico: Sistema de Estudiantes


In [ ]:
from sqlalchemy import create_engine, String, Float, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

# Nueva forma de definir la base
class Base(DeclarativeBase):
    pass

class Estudiante(Base):
    __tablename__ = 'estudiantes'
    
    # Con type annotations
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre: Mapped[str] = mapped_column(String(100), nullable=False)
    edad: Mapped[int] = mapped_column(Integer)
    curso: Mapped[str] = mapped_column(String(50))
    promedio: Mapped[float] = mapped_column(Float)
    
    def __repr__(self):
        return f"<Estudiante(nombre='{self.nombre}', curso='{self.curso}')>"

class SistemaEstudiantes:
    def __init__(self, nombre_db='estudiantes.db'):
        self.engine = create_engine(f'sqlite:///{nombre_db}', echo=False)
        Base.metadata.create_all(self.engine)
        print("✅ Base de datos creada")
    
    def agregar_estudiante(self, nombre, edad, curso, promedio):
        with Session(self.engine) as session:
            estudiante = Estudiante(
                nombre=nombre, 
                edad=edad, 
                curso=curso, 
                promedio=promedio
            )
            session.add(estudiante)
            session.commit()
            print(f"✅ Estudiante {nombre} agregado")
    
    def listar_estudiantes(self):
        with Session(self.engine) as session:
            stmt = select(Estudiante)
            estudiantes = session.scalars(stmt).all()
            
            if not estudiantes:
                print("📚 No hay estudiantes registrados")
                return
            
            print("👨‍🎓 Lista de estudiantes:")
            print("-" * 60)
            for est in estudiantes:
                print(f"ID: {est.id} | {est.nombre} ({est.edad} años)")
                print(f"   Curso: {est.curso} | Promedio: {est.promedio}")
                print("-" * 40)
    
    def buscar_por_curso(self, curso):
        with Session(self.engine) as session:
            stmt = select(Estudiante).where(Estudiante.curso == curso)
            estudiantes = session.scalars(stmt).all()
            
            if not estudiantes:
                print(f"❌ No hay estudiantes en el curso {curso}")
                return
            
            print(f"👥 Estudiantes del curso {curso}:")
            for est in estudiantes:
                print(f" - {est.nombre} (Promedio: {est.promedio})")
    
    def mejores_estudiantes(self, promedio_minimo=8.0):
        with Session(self.engine) as session:
            stmt = select(Estudiante).where(
                Estudiante.promedio >= promedio_minimo
            )
            estudiantes = session.scalars(stmt).all()
            
            if not estudiantes:
                print(f"🏆 No hay estudiantes con promedio >= {promedio_minimo}")
                return
            
            print(f"🏆 Mejores estudiantes (promedio >= {promedio_minimo}):")
            for est in estudiantes:
                print(f" - {est.nombre}: {est.promedio}")
    
    def actualizar_promedio(self, id_estudiante, nuevo_promedio):
        with Session(self.engine) as session:
            stmt = select(Estudiante).where(Estudiante.id == id_estudiante)
            estudiante = session.scalar(stmt)
            
            if estudiante:
                estudiante.promedio = nuevo_promedio
                session.commit()
                print(f"✅ Promedio de {estudiante.nombre} actualizado a {nuevo_promedio}")
            else:
                print("❌ Estudiante no encontrado")
    
    def eliminar_estudiante(self, id_estudiante):
        with Session(self.engine) as session:
            stmt = select(Estudiante).where(Estudiante.id == id_estudiante)
            estudiante = session.scalar(stmt)
            
            if estudiante:
                session.delete(estudiante)
                session.commit()
                print(f"✅ Estudiante {estudiante.nombre} eliminado")
            else:
                print("❌ Estudiante no encontrado")
    
    # Método adicional con ordenamiento
    def estudiantes_ordenados_por_promedio(self):
        from sqlalchemy import desc
        with Session(self.engine) as session:
            stmt = select(Estudiante).order_by(desc(Estudiante.promedio))
            estudiantes = session.scalars(stmt).all()
            
            print("📊 Estudiantes ordenados por promedio (descendente):")
            for est in estudiantes:
                print(f" - {est.nombre}: {est.promedio}")

# Probemos el sistema
if __name__ == "__main__":
    sistema = SistemaEstudiantes()
    
    # Agregar estudiantes de ejemplo
    sistema.agregar_estudiante("Ana García", 20, "Python", 8.5)
    sistema.agregar_estudiante("Carlos López", 22, "Python", 7.8)
    sistema.agregar_estudiante("María Fernández", 19, "Web", 9.2)
    sistema.agregar_estudiante("Pedro Martínez", 21, "Web", 8.9)
    sistema.agregar_estudiante("Laura Díaz", 23, "Datos", 9.5)
    
    print("\n" + "="*60)
    sistema.listar_estudiantes()
    
    print("\n" + "="*60)
    sistema.buscar_por_curso("Python")
    
    print("\n" + "="*60)
    sistema.mejores_estudiantes(8.5)
    
    print("\n" + "="*60)
    sistema.estudiantes_ordenados_por_promedio()
    
    print("\n" + "="*60)
    print("🔄 Actualizando promedio de Carlos...")
    sistema.actualizar_promedio(2, 8.7)  # ID de Carlos
    
    print("\n" + "="*60)
    sistema.mejores_estudiantes(8.5)

# 8.Ejemplo con relaciones de tablas y campos de fechas

- Tablas
    - `Usuario`: Información de usuarios (con fechas)
    - `Publicacion`: Publicaciones del blog (con fechas)
    - **Relación 1-N**: Un usuario puede tener muchas publicaciones
    - Clave foránea: autor_id en `Publicacion` referencia id en `Usuario`.

In [ ]:
from sqlalchemy import create_engine, String, Integer, Text, DateTime, ForeignKey, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session, relationship
from datetime import datetime, timedelta
from typing import List

# Nueva forma de definir la base
class Base(DeclarativeBase):
    pass

class Usuario(Base):
    __tablename__ = 'usuarios'
    
    id: Mapped[int] = mapped_column(primary_key=True)
    nombre: Mapped[str] = mapped_column(String(100), nullable=False)
    email: Mapped[str] = mapped_column(String(100), unique=True)
    fecha_registro: Mapped[datetime] = mapped_column(DateTime, default=datetime.now)
    ultimo_login: Mapped[datetime] = mapped_column(DateTime, nullable=True)
    
    # Relación uno-a-muchos con Publicacion
    publicaciones: Mapped[List["Publicacion"]] = relationship(back_populates="autor")
    
    def __repr__(self):
        return f"<Usuario(nombre='{self.nombre}', email='{self.email}')>"

class Publicacion(Base):
    __tablename__ = 'publicaciones'
    
    id: Mapped[int] = mapped_column(primary_key=True)
    titulo: Mapped[str] = mapped_column(String(200), nullable=False)
    contenido: Mapped[str] = mapped_column(Text)
    fecha_creacion: Mapped[datetime] = mapped_column(DateTime, default=datetime.now)
    fecha_actualizacion: Mapped[datetime] = mapped_column(
        DateTime, default=datetime.now, onupdate=datetime.now
    )
    
    # Clave foránea que relaciona con Usuario
    autor_id: Mapped[int] = mapped_column(ForeignKey('usuarios.id'))
    
    # Relación muchos-a-uno con Usuario
    autor: Mapped["Usuario"] = relationship(back_populates="publicaciones")
    
    def __repr__(self):
        return f"<Publicacion(titulo='{self.titulo}', autor_id={self.autor_id})>"

class BlogManager:
    def __init__(self, nombre_db='blog.db'):
        self.engine = create_engine(f'sqlite:///{nombre_db}', echo=False)
        Base.metadata.create_all(self.engine)
        print("✅ Base de datos del blog creada")
    
    def crear_usuario(self, nombre, email):
        with Session(self.engine) as session:
            usuario = Usuario(nombre=nombre, email=email)
            session.add(usuario)
            session.commit()
            print(f"✅ Usuario '{nombre}' creado")
            return usuario.id
    
    def crear_publicacion(self, titulo, contenido, autor_id):
        with Session(self.engine) as session:
            # Verificar que el autor existe
            stmt_autor = select(Usuario).where(Usuario.id == autor_id)
            autor = session.scalar(stmt_autor)
            
            if not autor:
                print("❌ Autor no encontrado")
                return
            
            publicacion = Publicacion(
                titulo=titulo,
                contenido=contenido,
                autor_id=autor_id
            )
            session.add(publicacion)
            session.commit()
            print(f"✅ Publicación '{titulo}' creada")
    
    def listar_usuarios(self):
        with Session(self.engine) as session:
            stmt = select(Usuario)
            usuarios = session.scalars(stmt).all()
            
            print("👥 Lista de usuarios:")
            print("-" * 50)
            for usuario in usuarios:
                print(f"ID: {usuario.id} | {usuario.nombre} ({usuario.email})")
                print(f"   Registrado: {usuario.fecha_registro.strftime('%d/%m/%Y %H:%M')}")
                if usuario.ultimo_login:
                    print(f"   Último login: {usuario.ultimo_login.strftime('%d/%m/%Y %H:%M')}")
                print(f"   Publicaciones: {len(usuario.publicaciones)}")
                print("-" * 30)
    
    def listar_publicaciones(self):
        with Session(self.engine) as session:
            stmt = select(Publicacion).join(Usuario)
            publicaciones = session.scalars(stmt).all()
            
            print("📝 Lista de publicaciones:")
            print("-" * 60)
            for pub in publicaciones:
                print(f"Título: {pub.titulo}")
                print(f"Autor: {pub.autor.nombre}")
                print(f"Creación: {pub.fecha_creacion.strftime('%d/%m/%Y %H:%M')}")
                print(f"Actualización: {pub.fecha_actualizacion.strftime('%d/%m/%Y %H:%M')}")
                print(f"Contenido: {pub.contenido[:50]}...")
                print("-" * 40)
    
    def publicaciones_recientes(self, dias=7):
        with Session(self.engine) as session:
            fecha_limite = datetime.now() - timedelta(days=dias)
            stmt = select(Publicacion).where(
                Publicacion.fecha_creacion >= fecha_limite
            ).join(Usuario).order_by(Publicacion.fecha_creacion.desc())
            
            publicaciones = session.scalars(stmt).all()
            
            print(f"🆕 Publicaciones de los últimos {dias} días:")
            for pub in publicaciones:
                print(f" - {pub.titulo} por {pub.autor.nombre}")
                print(f"   ({pub.fecha_creacion.strftime('%d/%m/%Y')})")
    
    def publicaciones_por_autor(self, autor_id):
        with Session(self.engine) as session:
            stmt_autor = select(Usuario).where(Usuario.id == autor_id)
            autor = session.scalar(stmt_autor)
            
            if not autor:
                print("❌ Autor no encontrado")
                return
            
            print(f"📚 Publicaciones de {autor.nombre}:")
            for pub in autor.publicaciones:
                print(f" - {pub.titulo}")
                print(f"   Creado: {pub.fecha_creacion.strftime('%d/%m/%Y')}")
                print(f"   Actualizado: {pub.fecha_actualizacion.strftime('%d/%m/%Y')}")
    
    def actualizar_ultimo_login(self, usuario_id):
        with Session(self.engine) as session:
            stmt = select(Usuario).where(Usuario.id == usuario_id)
            usuario = session.scalar(stmt)
            
            if usuario:
                usuario.ultimo_login = datetime.now()
                session.commit()
                print(f"✅ Último login de {usuario.nombre} actualizado")

# Ejemplo de uso
if __name__ == "__main__":
    blog = BlogManager()
    
    # Crear usuarios
    print("👤 Creando usuarios...")
    ana_id = blog.crear_usuario("Ana García", "ana@email.com")
    carlos_id = blog.crear_usuario("Carlos López", "carlos@email.com")
    maria_id = blog.crear_usuario("María Fernández", "maria@email.com")
    
    # Simular algunos logins
    blog.actualizar_ultimo_login(ana_id)
    blog.actualizar_ultimo_login(carlos_id)
    
    # Crear publicaciones
    print("\n📝 Creando publicaciones...")
    blog.crear_publicacion(
        "Mi primera publicación", 
        "Este es el contenido de mi primera publicación en el blog...",
        ana_id
    )
    blog.crear_publicacion(
        "Aprendiendo Python", 
        "Python es un lenguaje fantástico para desarrollo web...",
        carlos_id
    )
    blog.crear_publicacion(
        "Tutorial de SQLAlchemy", 
        "SQLAlchemy es un ORM muy poderoso para Python...",
        ana_id
    )
    blog.crear_publicacion(
        "Novedades en Desarrollo Web", 
        "Las últimas tendencias en desarrollo web para 2024...",
        maria_id
    )
    
    # Mostrar información
    print("\n" + "="*60)
    blog.listar_usuarios()
    
    print("\n" + "="*60)
    blog.listar_publicaciones()
    
    print("\n" + "="*60)
    blog.publicaciones_recientes(30)
    
    print("\n" + "="*60)
    blog.publicaciones_por_autor(ana_id)

## 9. Ejercicio propuesto
Crea un sistema para biblioteca con:
- Libros (título, autor, año, género)
- Buscar libros por autor o género
- Marcar libros como prestados

Una vez realizado, prueba a añadir la siguiente funcionalidad extra:
- Añade una tabla de Personas.
- Añade fechas de publicación en los Libros
- Añade una relación para gestionar el Préstamos de los Libros a las personas, guardando la fecha de préstamo

# SQLAlchemy 1.0 <span style="color:yellow">(deprecated)</span>

## 2. Configuración e instalación
Asegúrate de tener instalado SQLAlchemy:
```bash
pip install sqlalchemy
```

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

# Crear la base de datos
Base = declarative_base()

## 3. Definir nuestra primera tabla como clase Python

In [9]:
class Producto(Base):
    __tablename__ = 'productos'
    
    id = Column(Integer, primary_key=True)
    nombre = Column(String(100), nullable=False)
    precio = Column(Float, nullable=False)
    categoria = Column(String(50))
    
    def __repr__(self):
        return f"<Producto(nombre='{self.nombre}', precio={self.precio})>"

print("Clase Producto creada")

Clase Producto creada


## 4. Conectar y crear la base de datos

In [ ]:
# Crear motor de base de datos
engine = create_engine('sqlite:///tienda.db', echo=True)

# Crear todas las tablas
Base.metadata.create_all(engine)

# Crear sesión para trabajar
Session = sessionmaker(bind=engine)
session = Session()

print("Base de datos y sesión creadas")

## 5. Insertar datos (como objetos Python)

In [ ]:
productos_iniciales = [
    Producto(nombre="Laptop", precio=899.99, categoria="Tecnología"),
    Producto(nombre="Mouse", precio=25.50, categoria="Tecnología"),
    Producto(nombre="Libro Python", precio=35.00, categoria="Libros"),
    Producto(nombre="Café", precio=5.75, categoria="Alimentos")
]

# Agregar a la sesión y guardar
session.add_all(productos_iniciales)
session.commit()

print("Productos insertados en la base de datos")

## 6. Consultar datos

In [ ]:
# Obtener todos los productos
print("📦 Todos los productos:")
productos = session.query(Producto).all()
for producto in productos:
    print(f" - {producto.nombre}: ${producto.precio}")

# Consultas con filtros
print("\\n💻 Productos de tecnología:")
tecno = session.query(Producto).filter(Producto.categoria == "Tecnología").all()
for producto in tecno:
    print(f" - {producto.nombre}")

# Productos caros (más de $30)
print("\\n💰 Productos caros (> $30):")
caros = session.query(Producto).filter(Producto.precio > 30).all()
for producto in caros:
    print(f" - {producto.nombre}: ${producto.precio}")

## 7. Actualizar y eliminar

In [ ]:
# Actualizar un producto
laptop = session.query(Producto).filter(Producto.nombre == "Laptop").first()
if laptop:
    laptop.precio = 799.99  # ¡Oferta!
    session.commit()
    print(f"✅ Precio de {laptop.nombre} actualizado a ${laptop.precio}")

# Eliminar un producto
cafe = session.query(Producto).filter(Producto.nombre == "Café").first()
if cafe:
    session.delete(cafe)
    session.commit()
    print(f"✅ {cafe.nombre} eliminado")

# Verificar cambios
print("\\n Productos actualizados:")
productos_actualizados = session.query(Producto).all()
for producto in productos_actualizados:
    print(f" - {producto.nombre}: ${producto.precio}")

## 7.Ejemplo práctico: Sistema de Estudiantes

`sistema_estudiantes.py`

In [1]:
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

Base = declarative_base()

class Estudiante(Base):
    __tablename__ = 'estudiantes'
    
    id = Column(Integer, primary_key=True)
    nombre = Column(String(100), nullable=False)
    edad = Column(Integer)
    curso = Column(String(50))
    promedio = Column(Float)
    
    def __repr__(self):
        return f"<Estudiante(nombre='{self.nombre}', curso='{self.curso}')>"

class SistemaEstudiantes:
    def __init__(self, nombre_db='estudiantes.db'):
        self.engine = create_engine(f'sqlite:///{nombre_db}', echo=False)
        Base.metadata.create_all(self.engine)
        Session = sessionmaker(bind=self.engine)
        self.session = Session()
    
    def agregar_estudiante(self, nombre, edad, curso, promedio):
        estudiante = Estudiante(
            nombre=nombre, 
            edad=edad, 
            curso=curso, 
            promedio=promedio
        )
        self.session.add(estudiante)
        self.session.commit()
        print(f"✅ Estudiante {nombre} agregado")
    
    def listar_estudiantes(self):
        estudiantes = self.session.query(Estudiante).all()
        
        if not estudiantes:
            print("📚 No hay estudiantes registrados")
            return
        
        print("👨‍🎓 Lista de estudiantes:")
        print("-" * 60)
        for est in estudiantes:
            print(f"ID: {est.id} | {est.nombre} ({est.edad} años)")
            print(f"   Curso: {est.curso} | Promedio: {est.promedio}")
            print("-" * 40)
    
    def buscar_por_curso(self, curso):
        estudiantes = self.session.query(Estudiante).filter(
            Estudiante.curso == curso
        ).all()
        
        if not estudiantes:
            print(f"❌ No hay estudiantes en el curso {curso}")
            return
        
        print(f"👥 Estudiantes del curso {curso}:")
        for est in estudiantes:
            print(f" - {est.nombre} (Promedio: {est.promedio})")
    
    def mejores_estudiantes(self, promedio_minimo=8.0):
        estudiantes = self.session.query(Estudiante).filter(
            Estudiante.promedio >= promedio_minimo
        ).all()
        
        if not estudiantes:
            print(f"🏆 No hay estudiantes con promedio >= {promedio_minimo}")
            return
        
        print(f"🏆 Mejores estudiantes (promedio >= {promedio_minimo}):")
        for est in estudiantes:
            print(f" - {est.nombre}: {est.promedio}")
    
    def actualizar_promedio(self, id_estudiante, nuevo_promedio):
        estudiante = self.session.query(Estudiante).filter(
            Estudiante.id == id_estudiante
        ).first()
        
        if estudiante:
            estudiante.promedio = nuevo_promedio
            self.session.commit()
            print(f"✅ Promedio de {estudiante.nombre} actualizado a {nuevo_promedio}")
        else:
            print("❌ Estudiante no encontrado")
    
    def cerrar(self):
        self.session.close()

# Probemos el sistema
if __name__ == "__main__":
    sistema = SistemaEstudiantes()
    
    # Agregar estudiantes de ejemplo
    sistema.agregar_estudiante("Ana García", 20, "Python", 8.5)
    sistema.agregar_estudiante("Carlos López", 22, "Python", 7.8)
    sistema.agregar_estudiante("María Fernández", 19, "Web", 9.2)
    sistema.agregar_estudiante("Pedro Martínez", 21, "Web", 8.9)
    
    print("\\n" + "="*60)
    sistema.listar_estudiantes()
    
    print("\\n" + "="*60)
    sistema.buscar_por_curso("Python")
    
    print("\\n" + "="*60)
    sistema.mejores_estudiantes(8.5)
    
    sistema.cerrar()

ModuleNotFoundError: No module named 'sqlalchemy'